# Train an XGBoost model by manually implementing gradient boosting with weak learners, compare its accuracy with Random Forest, and analyze feature importance differences between both models.


In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Load Dataset
try:
    df = pd.read_csv("heart.csv")
except FileNotFoundError:
    df = pd.read_csv("heart.csv")

X = df.drop(columns=["HeartDisease"]).copy()
y = df["HeartDisease"].astype(int).values



In [3]:
# 2. Preprocessing
for col in X.columns:
    if pd.api.types.is_numeric_dtype(X[col]):
        X[col] = X[col].fillna(X[col].median())
    else:
        mode = X[col].mode()
        X[col] = X[col].fillna(
            mode.iloc[0] if not mode.empty else "Unknown"
        ).astype(str)

X = pd.get_dummies(X, dtype=float)
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

feature_names = X.columns.tolist()
X = X.to_numpy(dtype=float)

In [4]:
# 3. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

In [5]:

# 4. Random Forest Model
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

rf_accuracy = accuracy_score(y_test, rf_pred)
rf_importance = rf.feature_importances_



In [6]:
# 5. Manual Gradient Boosting - Weak Learners

def sigmoid(z):
    z = np.clip(z, -35, 35)
    return 1 / (1 + np.exp(-z))


def find_best_stump(X, gradient, hessian):
    reg_lambda = 1.0
    best_gain = 0
    best_stump = None

    total_g = gradient.sum()
    total_h = hessian.sum()

    for feature in range(X.shape[1]):
        values = X[:, feature]

        thresholds = np.unique(
            np.quantile(values, np.linspace(0.1, 0.9, 9))
        )

        for threshold in thresholds:
            left = values <= threshold
            right = ~left

            if left.sum() < 10 or right.sum() < 10:
                continue

            gl = gradient[left].sum()
            gr = gradient[right].sum()

            hl = hessian[left].sum()
            hr = hessian[right].sum()


In [7]:
# 5. Manual Gradient Boosting - Weak Learners

def sigmoid(z):
    z = np.clip(z, -35, 35)
    return 1 / (1 + np.exp(-z))


def find_best_stump(X, gradient, hessian):
    reg_lambda = 1.0
    best_gain = 0
    best_stump = None

    total_g = gradient.sum()
    total_h = hessian.sum()

    for feature in range(X.shape[1]):
        values = X[:, feature]

        thresholds = np.unique(
            np.quantile(values, np.linspace(0.1, 0.9, 9))
        )

        for threshold in thresholds:
            left = values <= threshold
            right = ~left

            if left.sum() < 10 or right.sum() < 10:
                continue

            gl = gradient[left].sum()
            gr = gradient[right].sum()

            hl = hessian[left].sum()
            hr = hessian[right].sum()

            # Gradient boosting split gain
            gain = 0.5 * (
                gl**2 / (hl + reg_lambda)
                + gr**2 / (hr + reg_lambda)
                - total_g**2 / (total_h + reg_lambda)
            )

            if gain > best_gain:
                best_gain = gain

                best_stump = {
                    "feature": feature,
                    "threshold": threshold,
                    "left_value": -gl / (hl + reg_lambda),
                    "right_value": -gr / (hr + reg_lambda),
                    "gain": gain
                }

    return best_stump


def train_manual_xgboost(X, y, rounds=60, learning_rate=0.1):
    # Initial prediction
    rate = np.clip(y.mean(), 1e-6, 1 - 1e-6)
    base_score = np.log(rate / (1 - rate))

    margin = np.full(len(y), base_score, dtype=float)

    learners = []
    importance = np.zeros(X.shape[1])

    for _ in range(rounds):
        probability = sigmoid(margin)

        # Calculate gradient and hessian
        gradient = probability - y
        hessian = np.maximum(
            probability * (1 - probability), 1e-6
        )

        # Find the best weak learner
        stump = find_best_stump(X, gradient, hessian)

        if stump is None:
            break

        learners.append(stump)

        feature = stump["feature"]
        left = X[:, feature] <= stump["threshold"]
        right = ~left

        # Update predictions using gradient boosting
        margin[left] += learning_rate * stump["left_value"]
        margin[right] += learning_rate * stump["right_value"]

        importance[feature] += stump["gain"]

    if importance.sum() > 0:
        importance /= importance.sum()

    model = {
        "base_score": base_score,
        "learners": learners,
        "learning_rate": learning_rate
    }

    return model, importance


def predict_manual_xgboost(model, X):
    margin = np.full(
        len(X), model["base_score"], dtype=float
    )

    for stump in model["learners"]:
        feature = stump["feature"]
        left = X[:, feature] <= stump["threshold"]
        right = ~left

        margin[left] += (
            model["learning_rate"] * stump["left_value"]
        )
        margin[right] += (
            model["learning_rate"] * stump["right_value"]
        )

    return (sigmoid(margin) >= 0.5).astype(int)


# 6. Train Manual XGBoost-style Model
xgb_model, xgb_importance = train_manual_xgboost(
    X_train, y_train
)

xgb_pred = predict_manual_xgboost(xgb_model, X_test)
xgb_accuracy = accuracy_score(y_test, xgb_pred)


# 7. Compare Accuracy
comparison = pd.DataFrame({
    "Model": ["Random Forest", "Manual XGBoost-Style"],
    "Accuracy (%)": [
        rf_accuracy * 100,
        xgb_accuracy * 100
    ]
})

print("MODEL ACCURACY COMPARISON")
print(comparison.round(2).to_string(index=False))


# 8. Compare Feature Importance
importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Random Forest Importance": rf_importance,
    "XGBoost Gain Importance": xgb_importance
})

print("\nTOP 10 RANDOM FOREST FEATURES")
print(
    importance_df.sort_values(
        "Random Forest Importance", ascending=False
    ).head(10).round(4).to_string(index=False)
)

print("\nTOP 10 XGBOOST-STYLE FEATURES")
print(
    importance_df.sort_values(
        "XGBoost Gain Importance", ascending=False
    ).head(10).round(4).to_string(index=False)
)

# 9. Analyze Differences
rf_top = set(
    importance_df.nlargest(
        5, "Random Forest Importance"
    )["Feature"]
)

xgb_top = set(
    importance_df.nlargest(
        5, "XGBoost Gain Importance"
    )["Feature"]
)

print("\nFEATURE IMPORTANCE ANALYSIS")
print("Common Top Features:", sorted(rf_top & xgb_top))
print("Random Forest Top Features:", sorted(rf_top))
print("XGBoost-Style Top Features:", sorted(xgb_top))

print("\nTASK COMPLETED!")

MODEL ACCURACY COMPARISON
               Model  Accuracy (%)
       Random Forest         90.22
Manual XGBoost-Style         88.59

TOP 10 RANDOM FOREST FEATURES
          Feature  Random Forest Importance  XGBoost Gain Importance
      ST_Slope_Up                    0.1519                   0.5419
          Oldpeak                    0.0983                   0.0052
    ST_Slope_Flat                    0.0949                   0.0000
      Cholesterol                    0.0941                   0.0806
ChestPainType_ASY                    0.0937                   0.2155
            MaxHR                    0.0931                   0.0079
              Age                    0.0696                   0.0022
        RestingBP                    0.0648                   0.0000
 ExerciseAngina_N                    0.0529                   0.0875
 ExerciseAngina_Y                    0.0507                   0.0000

TOP 10 XGBOOST-STYLE FEATURES
          Feature  Random Forest Importance  XGB